# Model 2: Fine-tuned Model
### ⚠ Run this notebook in Google Colab with a GPU (Runtime → Change runtime type → T4 GPU)

**Goal:** Fine-tune a German GPT-2 model on Austrian tax law Q&A examples, then run it on 644 questions.

**3 steps:**
1. Generate training data (use OpenAI API to create ~150 Q&A examples)
2. Fine-tune `dbmdz/german-gpt2` on those examples using HuggingFace Trainer
3. Run the fine-tuned model on all 644 questions and download results

In [ ]:
# Install required packages
!pip install openai transformers datasets pandas torch --quiet

In [ ]:
import json
import pandas as pd
import torch
from openai import OpenAI
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Check GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ⚠ Replace with your API key
API_KEY = "INSERT-YOUR-KEY-HERE"
client = OpenAI(api_key=API_KEY)

## Step 1: Generate Training Data

We use the OpenAI API to generate ~150 Austrian tax law Q&A pairs.
These will be used to teach our German GPT-2 model.

In [ ]:
TOPICS = [
    "Körperschaftsteuer: Bemessungsgrundlage, Steuersatz, unbeschränkte Steuerpflicht (KStG)",
    "Körperschaftsteuer: verdeckte Ausschüttung, Beteiligungserträge (KStG § 8, § 10)",
    "Körperschaftsteuer: Verlustabzug, Verlustvorträge, Mantelkauf (KStG § 8, § 12)",
    "Gruppenbesteuerung: Gruppenträger, Gruppenmitglieder, Verlustzurechnung (KStG § 9)",
    "Einkommensteuer: Selbstständige, Gewerbebetrieb, Betriebsausgaben (EStG)",
    "Einkommensteuer: Arbeitnehmer, Lohnsteuer, Sonderausgaben (EStG)",
    "Internationales Steuerrecht: Doppelbesteuerung, Ansässigkeit, Betriebsstätte",
    "Internationales Steuerrecht: Holdinggesellschaften, Hinzurechnungsbesteuerung",
    "Umsatzsteuer: Steuerpflicht, Vorsteuerabzug, Reverse-Charge (UStG)",
    "Körperschaftsteuer: Gesellschafterdarlehen, Forderungsverzicht (KStG)"
]

PROMPT = """Erstelle genau 15 Frage-Antwort-Paare zum österreichischen Steuerrecht für: {topic}

Gib NUR ein JSON-Array zurück:
[{{"question": "Frage", "answer": "Antwort mit § Verweis"}}]

Antworten: Deutsch, 1–3 Sätze, mit Paragraphenverweis (z.B. § 7 Abs. 1 KStG)."""

qa_pairs = []

for i, topic in enumerate(TOPICS):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": PROMPT.format(topic=topic)}],
        max_completion_tokens=2000,
        temperature=0.7
    )
    raw = response.choices[0].message.content.strip()
    # Strip markdown code blocks if GPT wraps the JSON in ```json ... ```
    raw = raw.strip("```json").strip("```").strip()
    try:
        pairs = json.loads(raw)
        qa_pairs.extend(pairs)
        print(f"Topic {i+1}/{len(TOPICS)}: {len(pairs)} pairs ✓")
    except Exception as e:
        print(f"Topic {i+1}/{len(TOPICS)}: failed ({e})")

print(f"\nTotal training pairs: {len(qa_pairs)}")

In [ ]:
# Format training data as text: "Frage: X\nAntwort: Y"
# This is the standard format for causal language model fine-tuning
texts = [f"Frage: {p['question']}\nAntwort: {p['answer']}" for p in qa_pairs]

train_dataset = Dataset.from_dict({"text": texts})
print(f"Training examples: {len(train_dataset)}")
print("\nExample:")
print(texts[0])

## Step 2: Fine-tune the Model

We fine-tune `dbmdz/german-gpt2` — a GPT-2 model pre-trained on German text.
Same approach as `lecture_3_sft.ipynb` but for text generation instead of classification.

In [ ]:
# Load tokenizer and model
model_name = "dbmdz/german-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
print(f"Model loaded: {model_name}")

In [ ]:
# Tokenize the training data
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
# Note: no set_format("torch") — DataCollatorForLanguageModeling handles this automatically

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(f"Tokenized {len(tokenized_dataset)} examples")

In [ ]:
# Define training arguments (same style as lecture_3_sft.ipynb)
training_args = TrainingArguments(
    output_dir="./finetuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=20,
    save_strategy="no",
    report_to="none"        # disables Weights & Biases login prompt
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Train the model
trainer.train()

In [ ]:
# Save the fine-tuned model
trainer.save_model("./finetuned_model")
tokenizer.save_pretrained("./finetuned_model")
print("Model saved to ./finetuned_model")

## Step 3: Run Inference on 644 Questions

We load the dataset directly from GitHub and generate answers using our fine-tuned model.

In [ ]:
# Load dataset directly from GitHub (no file upload needed in Colab)
df = pd.read_csv("https://raw.githubusercontent.com/frederrrrr/wu-llms-ss26/main/dataset_clean.csv")
print(f"Loaded {len(df)} questions")
df.head(3)

In [ ]:
# Load fine-tuned model for inference
ft_tokenizer = AutoTokenizer.from_pretrained("./finetuned_model")
ft_model = AutoModelForCausalLM.from_pretrained("./finetuned_model").to(device)
ft_model.eval()
print("Fine-tuned model loaded")

In [ ]:
# Generate answers for all 644 questions
answers = []

for i, row in df.iterrows():
    prompt = f"Frage: {row['prompt']}\nAntwort:"
    inputs = ft_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=200).to(device)

    with torch.no_grad():
        output_ids = ft_model.generate(
            **inputs,
            max_new_tokens=150,
            pad_token_id=ft_tokenizer.eos_token_id,
            do_sample=False
        )

    full_text = ft_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Extract only the answer part (everything after "Antwort:")
    answer = full_text.split("Antwort:")[-1].strip()
    answers.append(answer)
    print(f"{i+1}/{len(df)} | {row['id']} | {answer[:70]}...")

In [ ]:
# Save results CSV
results = pd.DataFrame({"id": df["id"], "answer": answers})
results.to_csv("model2_results.csv", index=False)
print(f"Saved {len(results)} answers")
results.head(3)

In [ ]:
# Download the results CSV to your computer
from google.colab import files
files.download("model2_results.csv")
print("Download started — move the file to VAT-INTL-001/results/")